<a href="https://colab.research.google.com/github/naveelsingh10/Solid_Ai_Hackathon/blob/main/Solid_Ai_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

#MAKE CHANGES IN TEST DIR ACC TO YOUR#REQUIREMENTS (UPDATE THE TEST DIRECTORY PATH)
test_dir = '/content/drive/MyDrive/test'  # Update this path to your test directory


import os



os.system('pip install ultralytics pandas huggingface_hub')

import pandas as pd
from ultralytics import YOLO
from collections import Counter
from huggingface_hub import hf_hub_download

repo_id = "naveelsingh10/final_solidworks-yolo"
hf_filename = "weights/best.pt"

output_filename = 'submission.csv'
target_columns = ['bolt', 'locatingpin', 'nut', 'washer']

print(f"Downloading model from {repo_id}...")

try:
    model_path = hf_hub_download(repo_id=repo_id, filename=hf_filename)
except Exception:
    model_path = hf_hub_download(repo_id=repo_id, filename="best.pt")

print(f"Model downloaded to {model_path}")

model = YOLO(model_path)

print("Starting inference...")

results_generator = model.predict(
    source=test_dir,
    conf=0.25,
    stream=True,
    verbose=False
)

data = []

for result in results_generator:
    full_path = result.path
    file_name = os.path.basename(full_path)

    class_indices = result.boxes.cls.cpu().numpy().astype(int)
    detected_names = [model.names[i] for i in class_indices]

    counts = Counter(detected_names)

    row = {'image_name': file_name}
    for col in target_columns:
        row[col] = counts.get(col, 0)

    data.append(row)

df = pd.DataFrame(data)

df = df[['image_name'] + target_columns]
df = df.sort_values('image_name')

df.to_csv(output_filename, index=False)

print("Submission file created successfully.")
print(df.head())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


weights/best.pt:   0%|          | 0.00/40.6M [00:00<?, ?B/s]

Model downloaded to /root/.cache/huggingface/hub/models--naveelsingh10--final_solidworks-yolo/snapshots/b73744d41c6a45439c8c01679a486e11717b158d/weights/best.pt
Starting inference...
Submission file created successfully.
                             image_name  bolt  locatingpin  nut  washer
0  0040313aa7c7478c8ca264bf573c53fe.png     1            3    0       0
1  0080e45f4375443b96ee81dc04075117.png     2            0    0       1
2  0086da1a1914469caa4d042f8e4e94ef.png     0            0    0       1
3  009ccd5b64a043e0afbae0f840f7cfd4.png     0            0    0       1
4  00a522a027b24fcfb8ee0857655b953d.png     0            1    0       0
